# Daily Challenge: How to Finetune LLMs with LoRA
Learn how to apply Low-Rank Adaptation (LoRA) for parameter-efficient fine-tuning of large language models using Hugging Face PEFT.

In [ ]:
# Install required libraries
%pip install peft==0.4.0
%pip install datasets

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_name = 'bigscience/bloomz-560m'
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
# Load and preprocess the dataset
data = load_dataset('Abirate/english_quotes', split='train[:10%]')
data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)
train_sample = data.select(range(5))
display(train_sample)

In [ ]:
import peft
from peft import LoraConfig, get_peft_model
lora_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=['query_key_value', 'dense'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)
peft_model = get_peft_model(foundation_model, lora_config)
print(peft_model.print_trainable_parameters())

In [ ]:
import transformers
from transformers import TrainingArguments, Trainer
import os
output_directory = os.path.join('cache', 'peft_lab_outputs')
training_args = TrainingArguments(
    report_to='none',
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,
    num_train_epochs=1,
    use_cpu=True
)
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=data,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
trainer.train()

In [ ]:
import time
time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f'peft_model_{time_now}')
trainer.model.save_pretrained(peft_model_path)

In [ ]:
from peft import PeftModel
loaded_peft_model = PeftModel.from_pretrained(foundation_model, peft_model_path, is_trainable=False)

In [ ]:
inputs = tokenizer('Two things are infinite: ', return_tensors='pt')
outputs = loaded_peft_model.generate(
    input_ids=inputs['input_ids'],
    attention_mask=inputs['attention_mask'],
    max_new_tokens=30,
    do_sample=True,
    top_k=50,
    top_p=0.95
)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True))